In [17]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from DATA.stock_invest_function import *


In [18]:
def calculate_correlation_between_dfs(df1, df2, start_date=None, end_date=None, method='pearson', min_periods=4):
    """
    두 개의 시계열 DataFrame의 상관관계를 계산하되, 유효 관측치가 min_periods보다 많을 경우만 수행

    Parameters:
    ...
    - min_periods (int): 최소 유효 데이터 수

    Returns:
    - pd.DataFrame: 상관계수 매트릭스
    """
    if start_date:
        df1 = df1[df1.index >= pd.to_datetime(start_date)]
        df2 = df2[df2.index >= pd.to_datetime(start_date)]
    if end_date:
        df1 = df1[df1.index <= pd.to_datetime(end_date)]
        df2 = df2[df2.index <= pd.to_datetime(end_date)]

    combined = pd.merge(df1, df2, left_index=True, right_index=True, how='inner', suffixes=('_firm', '_hs'))

    corr_matrix = pd.DataFrame(index=df1.columns, columns=df2.columns, dtype=float)

    for firm in df1.columns:
        for hs in df2.columns:
            x = combined[firm]
            y = combined[hs]
            valid = x.notna() & y.notna()
            if valid.sum() >= min_periods:
                corr_matrix.loc[firm, hs] = x[valid].corr(y[valid], method=method)
            else:
                corr_matrix.loc[firm, hs] = np.nan  # 또는 0

    return corr_matrix

def get_top_correlated_hscode(corr_matrix, symbol, top_n=5, threshold=None, ascending=False):
    """
    특정 기업(Symbol)에 대해 상관관계가 높은 HS 코드를 추출하는 함수

    Parameters:
    - corr_matrix (pd.DataFrame): Symbol x HS_Code 형태의 상관관계 행렬
    - symbol (str): 대상 Symbol (예: '000080')
    - top_n (int): 상위 N개 추출 (threshold와 함께 사용 시 무시될 수 있음)
    - threshold (float or None): 상관계수 하한값 (예: 0.5 이상만 보기). 설정 시 top_n보다 우선함
    - ascending (bool): 상관계수 기준 오름차순 정렬 여부 (기본값: False = 높은 값 우선)

    Returns:
    - pd.DataFrame: root_hs_code 및 상관계수를 포함한 상위 N개 HS 코드
    """

    if symbol not in corr_matrix.index:
        raise ValueError(f"Symbol '{symbol}' not found in correlation matrix.")

    symbol_corr = corr_matrix.loc[symbol].dropna()

    if threshold is not None:
        symbol_corr = symbol_corr[symbol_corr >= threshold]

    top_correlated = symbol_corr.sort_values(ascending=ascending).head(top_n)

    return top_correlated.reset_index().rename(columns={'index': 'root_hs_code', symbol: 'correlation'})

def get_top_correlated_symbols(corr_matrix, hs_code, top_n=5, threshold=None, ascending=False):
    """
    특정 HS 코드에 대해 상관관계가 높은 기업 Symbol을 추출하는 함수

    Parameters:
    - corr_matrix (pd.DataFrame): Symbol x HS_Code 형태의 상관관계 행렬
    - hs_code (str or int): 대상 HS 코드 (예: '151550')
    - top_n (int): 상위 N개 추출
    - threshold (float or None): 상관계수 하한값 (예: 0.5 이상만 보기)
    - ascending (bool): 정렬 방향 (False: 높은 상관 우선)

    Returns:
    - pd.DataFrame: symbol 및 correlation 정보를 담은 상위 N개 결과
    """

    if hs_code not in corr_matrix.columns:
        raise ValueError(f"HS code '{hs_code}' not found in correlation matrix columns.")

    hs_corr = corr_matrix[hs_code].dropna()

    if threshold is not None:
        hs_corr = hs_corr[hs_corr >= threshold]

    top_symbols = hs_corr.sort_values(ascending=ascending).head(top_n)

    return top_symbols.reset_index().rename(columns={'index': 'symbol', hs_code: 'correlation'})


In [19]:
db_info = {
    'host': get_db_host(),
    # 'host': '192.168.0.230',
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

# SQLAlchemy 엔진 생성
engine = create_engine(
    f"mysql+pymysql://{db_info['user']}:{db_info['password']}@{db_info['host']}:{db_info['port']}/{db_info['database']}"
)

# 테이블 이름
table_name = 'target_hs_code'

# 고유한 hs_code 값 추출 쿼리 실행
query = f"SELECT DISTINCT hs_code FROM {table_name}"
unique_hs_codes_df = pd.read_sql(query, con=engine)
hs_codes  = unique_hs_codes_df['hs_code'].unique().tolist()

indicator = 'expDlr'

df_real = fetch_trade_data_multi_hscode(db_info, hs_codes, indicator)

# 분기 정보 추가
df_real['quarter'] = df_real['date'].dt.to_period('Q')

# 그룹별로 분기별 합산
df_quarterly = (
    df_real
    .groupby(['root_hs_code', 'quarter'])['value']
    .sum()
    .reset_index()
)

# 👉 분기 월말로 변환 (예: 2007Q1 → 2007-03-31)
df_quarterly['date'] = df_quarterly['quarter'].dt.to_timestamp(how='end')

# 👉 'quarter' 컬럼 제거
df_quarterly.drop(columns=['quarter'], inplace=True)

# 1단계: 문자열로 직접 변환하려면 to_datetime 이후에 바로 strftime
df_quarterly['date'] = pd.to_datetime(df_quarterly['date']).dt.strftime('%Y-%m-%d')

def create_yoy_growth_pivot(df_quarterly, start_date=None, end_date=None):
    """
    전년 동분기 대비 증가율을 pivot 형태로 변환하고 분석기간을 설정할 수 있는 함수

    Parameters:
    - df_quarterly (DataFrame): 'root_hs_code', 'date', 'yoy_growth' 포함된 데이터
    - start_date (str or None): 분석 시작일 (예: '2015-01-01')
    - end_date (str or None): 분석 종료일 (예: '2023-12-31')

    Returns:
    - pivot_df (DataFrame): 행: date, 열: root_hs_code, 값: yoy_growth
    """
    # Pivot
    pivot_df = df_quarterly.pivot(
        index='date',
        columns='root_hs_code',
        values='yoy_growth'
    ).sort_index()

    # inf 값 NaN 처리
    pivot_df.replace([np.inf, -np.inf], np.nan, inplace=True)

    # 분석 기간 슬라이싱 (날짜가 문자열이면 datetime으로 변환)
    pivot_df.index = pd.to_datetime(pivot_df.index)

    if start_date:
        pivot_df = pivot_df[pivot_df.index >= pd.to_datetime(start_date)]
    if end_date:
        pivot_df = pivot_df[pivot_df.index <= pd.to_datetime(end_date)]

    return pivot_df


# 전년 동분기 값 (4개 분기 전 값) 계산
df_quarterly['yoy_value'] = (
    df_quarterly
    .sort_values(['root_hs_code', 'date'])
    .groupby('root_hs_code')['value']
    .shift(4)
)

# ❗ yoy_growth 계산
df_quarterly['yoy_growth'] = (
    (df_quarterly['value'] - df_quarterly['yoy_value']) / df_quarterly['yoy_value']
) * 100

quarterly_trade_data = create_yoy_growth_pivot(df_quarterly, start_date='2008-03', end_date='2025-03')

In [20]:
fs_df = fetch_table_data(db_info, "korea_fs_data")
fs_df.rename(columns={'Date': 'date'}, inplace=True)

# 1. indicator 필터링
target_indicator = '매출액(천원)'
filtered_df = fs_df[fs_df['indicator'] == target_indicator].copy()

# 2. 날짜 정제 및 정렬
filtered_df['date'] = pd.to_datetime(filtered_df['date'])
filtered_df.sort_values(by='date', inplace=True)

# 3. value 컬럼이 있는지 확인 및 타입 강제
if 'value' not in filtered_df.columns:
    raise KeyError("'value' 컬럼이 없습니다.")

filtered_df['value'] = pd.to_numeric(filtered_df['value'], errors='coerce')

# 4. 피벗 테이블 생성 (행: date, 열: Symbol, 값: value)
pivot_df = filtered_df.pivot_table(
    index='date',
    columns='symbol',
    values='value',
    aggfunc='first'  # 중복 방지
)

# 5. 전년 동분기 대비 변화율 계산 (4분기 전 대비)
fs_yoy_growth_df = pivot_df.pct_change(periods=4) * 100

✅ 'korea_fs_data' 테이블에서 5584577건의 데이터를 가져왔습니다.


In [21]:
correlation_result = calculate_correlation_between_dfs(
    fs_yoy_growth_df,
    quarterly_trade_data,
    start_date='2020-03-31',
    end_date='2025-03-31'
)

# 상위 몇 개 확인
correlation_result.head()

root_hs_code,121120,1212,121221,151550,151590,1518,170199,1902,190230,1905,...,903289,9301,9306,9401,940130,940199,940330,940540,950300,970191
symbol,,,,,,,,,,,,,,,,,,,,,
A000010,0.295863,0.264440,0.264535,-0.405793,-0.047399,-0.535232,0.113347,-0.438034,-0.355935,-0.774895,...,-0.000400,0.205490,-0.453902,-0.108215,-0.629688,-0.853721,0.395599,-0.834817,-0.091785,0.732864
A000020,0.070134,0.537656,0.536061,-0.176234,0.693954,-0.186943,-0.474600,-0.138111,-0.022520,-0.411699,...,-0.092314,-0.202735,-0.245582,0.299020,-0.516051,0.429429,0.382900,-0.587345,0.428124,-0.086697
A000030,0.303578,0.276828,0.276907,-0.412560,-0.037144,-0.560932,0.091884,-0.458652,-0.374004,-0.779881,...,0.003928,0.171950,-0.472233,-0.080982,-0.643218,-0.864607,0.418791,-0.837847,-0.067112,0.749627
A000040,0.155914,-0.172157,-0.175293,0.271092,-0.015489,0.051555,-0.241148,-0.241248,-0.221853,-0.026255,...,-0.469398,0.126598,0.233490,0.079769,-0.240429,-0.071856,0.010413,0.305215,-0.099375,0.252328
A000050,-0.096418,0.027677,0.026816,0.037008,-0.056782,-0.213419,0.031239,0.117152,0.109311,0.388668,...,-0.012355,-0.704000,-0.288004,0.222897,-0.218923,0.376524,0.284617,0.244170,0.230996,-0.532924


In [23]:
correlation_result

root_hs_code,121120,1212,121221,151550,151590,1518,170199,1902,190230,1905,...,903289,9301,9306,9401,940130,940199,940330,940540,950300,970191
symbol,,,,,,,,,,,,,,,,,,,,,
A000010,0.295863,0.264440,0.264535,-0.405793,-0.047399,-0.535232,0.113347,-0.438034,-0.355935,-0.774895,...,-0.000400,0.205490,-0.453902,-0.108215,-0.629688,-0.853721,0.395599,-0.834817,-0.091785,0.732864
A000020,0.070134,0.537656,0.536061,-0.176234,0.693954,-0.186943,-0.474600,-0.138111,-0.022520,-0.411699,...,-0.092314,-0.202735,-0.245582,0.299020,-0.516051,0.429429,0.382900,-0.587345,0.428124,-0.086697
A000030,0.303578,0.276828,0.276907,-0.412560,-0.037144,-0.560932,0.091884,-0.458652,-0.374004,-0.779881,...,0.003928,0.171950,-0.472233,-0.080982,-0.643218,-0.864607,0.418791,-0.837847,-0.067112,0.749627
A000040,0.155914,-0.172157,-0.175293,0.271092,-0.015489,0.051555,-0.241148,-0.241248,-0.221853,-0.026255,...,-0.469398,0.126598,0.233490,0.079769,-0.240429,-0.071856,0.010413,0.305215,-0.099375,0.252328
A000050,-0.096418,0.027677,0.026816,0.037008,-0.056782,-0.213419,0.031239,0.117152,0.109311,0.388668,...,-0.012355,-0.704000,-0.288004,0.222897,-0.218923,0.376524,0.284617,0.244170,0.230996,-0.532924
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
A950180,0.307675,0.244503,0.247202,-0.544369,0.115495,-0.419966,-0.082620,-0.568258,-0.565651,-0.455475,...,0.249307,0.053724,-0.294078,0.202557,-0.221289,-0.578113,-0.108230,-0.424662,0.055184,0.822625
A950190,0.078970,-0.130669,-0.132688,-0.023233,0.052983,-0.109114,-0.029406,-0.126134,-0.107176,-0.296493,...,0.165996,-0.074139,-0.138058,0.051541,0.096063,-0.319711,0.815474,-0.388836,0.004922,0.085886
A950200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.548365,NaN,NaN,NaN,-0.235305


In [24]:
 correlation_result.to_csv(r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA\한국상장사_수출데이터_상관계수_202508.csv", index=True, encoding="utf-8-sig")

# path = r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA\한국상장사_수출데이터_상관계수.csv"
path = r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA\한국상장사_수출데이터_상관계수_202508.csv"
correlation_result = pd.read_csv(path).set_index('symbol')

In [8]:
top_hs_codes = get_top_correlated_hscode(
    corr_matrix=correlation_result,  # 이전에 만든 상관관계 행렬
    symbol ='A066590',
    top_n=50,
    threshold=0.3  # 선택사항
)

print(top_hs_codes)

   root_hs_code  correlation
0        848210     0.943980
1        902213     0.920762
2        292910     0.917066
3        390950     0.916766
4        550320     0.905617
5        848390     0.899995
6        870850     0.896875
7        848350     0.891924
8        390390     0.891740
9        851290     0.889384
10       840991     0.886672
11       870840     0.883461
12       560394     0.879690
13       820900     0.876990
14       846729     0.876170
15       722592     0.875567
16       392690     0.874234
17       722530     0.872893
18       731816     0.872597
19       722550     0.867143
20       848790     0.863249
21       854420     0.860585
22       400270     0.859940
23       848220     0.859171
24       901849     0.857787
25       790112     0.853275
26       870880     0.853084
27       848299     0.850068
28       851140     0.849325
29       870899     0.844960
30       950300     0.844607
31       845020     0.841352
32       870830     0.838694
33       79012

In [9]:
from pykrx import stock

# 1. ticker 리스트 불러오기
tickers = stock.get_market_ticker_list(market="ALL")

# 2. ticker와 name을 리스트로 만들기
data = []
for t in tickers:
    name = stock.get_market_ticker_name(t)
    data.append({
        'ticker': t,
        'name': name,
        'symbol': 'A' + t
    })

# 3. DataFrame으로 변환
company_name_df = pd.DataFrame(data, columns=['symbol', 'ticker', 'name'])


In [26]:
top_symbols = get_top_correlated_symbols(
    corr_matrix=correlation_result,
    hs_code= '8535',
    top_n= 50,
    threshold=0.1  # 선택사항
)
print(top_symbols)

     symbol  correlation
0   A078860     0.698684
1   A123840     0.694974
2   A227610     0.691114
3   A004650     0.671981
4   A053980     0.663863
5   A221610     0.637982
6   A293780     0.628025
7   A021050     0.610933
8   A009680     0.608342
9   A010660     0.603160
10  A005440     0.592462
11  A051360     0.585613
12  A009540     0.577711
13  A017510     0.574619
14  A004150     0.571774
15  A101240     0.566197
16  A028040     0.564263
17  A071200     0.553791
18  A032620     0.553490
19  A101390     0.551470
20  A265560     0.551274
21  A002790     0.551048
22  A040610     0.549941
23  A049630     0.548112
24  A003570     0.546952
25  A052260     0.545701
26  A205500     0.543129
27  A237750     0.540832
28  A032980     0.539240
29  A012170     0.536338
30  A038620     0.536142
31  A003850     0.535785
32  A204990     0.533741
33  A090150     0.532230
34  A214330     0.530058
35  A011150     0.529242
36  A104040     0.525595
37  A195500     0.522311
38  A073490     0.522282


In [11]:
correlation_result.loc['A131290'][['903090']]

903090    0.630714
Name: A131290, dtype: float64

In [12]:
pd.merge(top_symbols, company_name_df, on='symbol', how = 'left')

,symbol,correlation,ticker,name
0,A206560,0.803367,206560,덱스터
1,A123570,0.799098,123570,이엠넷
2,A060720,0.787006,060720,KH바텍
3,A018290,0.778518,018290,브이티
4,A310200,0.759977,310200,애니플러스
...,...,...,...,...
65,A036930,0.639544,036930,주성엔지니어링
66,A001250,0.636846,001250,GS글로벌
67,A033430,0.635122,NaN,NaN
68,A112240,0.634576,NaN,NaN


In [13]:
hscode = fetch_table_data(db_info, "target_hs_code")

hscode[hscode['hs_code'] == '854232']

✅ 'target_hs_code' 테이블에서 567건의 데이터를 가져왔습니다.


,hs_code
417,854232


In [14]:
name

'힘스'